# From URDF to a Semantic Digital Twin

*IJCAI 2026 workshop — hands-on tutorial (90 minutes)*

A URDF tells a robot **where** every link is. It does not tell the robot **what** any of
them is. `<link name="cabinet6_drawer_top"/>` is a name a human chose; to the robot it is a
rigid body on a prismatic joint, indistinguishable from a sliding door or a telescopic mast.

So "open the drawer" is not a question a URDF can answer.

This tutorial closes that gap using the
[Semantic Digital Twin](https://github.com/cram2/cognitive_robot_abstract_machine), a world
model that carries geometry, kinematics **and** meaning in one structure.

| § | What we do |
|---|---|
| 1 | Load an apartment URDF and watch two reasonable heuristics both get it wrong |
| 2 | Say what things *are*: build a dresser from typed semantic annotations |
| 3 | Let the `WorldReasoner` find the drawers itself — and explain why |
| 4 | Ask the world questions with the Entity Query Language |
| 5 | Find what the reasoner is still missing, and teach it the rule that fixes it |

**Prerequisites:** Python, and having seen a URDF before. No prior knowledge of CRAM,
ontologies, or rule-based reasoning is assumed.

## 0. Setup

> **Check your kernel.** The top right of this notebook must say **CRAM**. If it says
> anything else, use *Kernel → Change Kernel… → CRAM*. The default Python kernel does not
> have the semantic digital twin installed.

Run the cell below. It should print a version and two `OK` lines.

In [1]:
import logging
from pathlib import Path
from importlib.resources import files

logging.disable(logging.CRITICAL)  # keep the notebook output readable

import semantic_digital_twin
from semantic_digital_twin.adapters.package_resolver import CompositePathResolver

print("semantic_digital_twin", semantic_digital_twin.__version__)

URDF_DIR = Path(files("semantic_digital_twin")).parent.parent / "resources" / "urdf"
APARTMENT = URDF_DIR / "apartment.urdf"
print("OK  apartment URDF:", APARTMENT.name)

# The apartment references its meshes as package://iai_apartment/... , which comes from the
# iai_maps ROS package. If this line fails, the ROS workspace was not sourced.
CompositePathResolver().resolve("package://iai_apartment/meshes/visual/walls.dae")
print("OK  mesh packages resolve")

semantic_digital_twin 26.07.0
OK  apartment URDF: apartment.urdf
OK  mesh packages resolve


## 1. What a URDF can and cannot tell a robot

`URDFParser` reads a URDF into a `World`. A `World` is a graph: bodies and regions as
nodes, connections (joints) as edges, with a registry of degrees of freedom.

*(This URDF puts a `material` tag inside a `collision` element, which is not legal there, so
the XML parser prints an `Unknown tag` warning. It is harmless — real-world URDFs are rarely
clean.)*

In [2]:
from collections import Counter
from semantic_digital_twin.adapters.urdf import URDFParser

world = URDFParser.from_file(str(APARTMENT)).parse()

print("bodies     ", len(world.bodies))
print("connections", len(world.connections))
print(Counter(type(c).__name__ for c in world.connections))

Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]


bodies      113
connections 112
Counter({'FixedConnection': 74, 'PrismaticConnection': 25, 'RevoluteConnection': 13})


Let's look at it. `RayTracer` renders the world straight into the notebook — no simulator,
no ROS, no window manager.

*(Drag to orbit, scroll to zoom.)*

In [3]:
from semantic_digital_twin.spatial_computations.raytracer import RayTracer

ray_tracer = RayTracer(world)
ray_tracer.update_scene()
ray_tracer.scene.show("jupyter")

### Now the important part

Ask the world what it *means*:

In [4]:
print(world.semantic_annotations)

[]


Empty. A world parsed from a file is a purely kinematic model. Nothing in it is a drawer, a
handle, or a door — those are things *we* read into the link names.

### Exercise 1 — find everything the robot can open

You are writing the perception layer for a robot in this apartment. It needs a list of
drawers. There are two obvious ways to guess, and they are both reasonable.

**Guess A — match the names.** Somebody called them drawers, so search for that:

In [5]:
by_name = {body.name.name for body in world.bodies if "drawer" in body.name.name.lower()}
print(len(by_name), "bodies matched by name")

25 bodies matched by name


**Guess B — match the structure.** A drawer slides, so look for prismatic joints:

In [6]:
from semantic_digital_twin.world_description.connections import PrismaticConnection

sliding = {c.child.name.name for c in world.connections if isinstance(c, PrismaticConnection)}
print(len(sliding), "bodies sit on a prismatic joint")

25 bodies sit on a prismatic joint


Both say 25. That looks like confirmation — two independent methods agreeing.

It isn't. Compare the actual sets:

In [7]:
print("both agree on   :", len(by_name & sliding))
print()
print("named, not sliding:", sorted(by_name - sliding))
print("sliding, not named:", sorted(sliding - by_name))

both agree on   : 23

named, not sliding: ['handle_cab1_drawer_bottom', 'handle_cab1_drawer_mid']
sliding, not named: ['cabinet2_door_out_fancy', 'cabinet3_door_bottom_out_fancy']


The two heuristics agree on a *number* and disagree about *which bodies*, and each is wrong
in its own way:

- `handle_cab1_drawer_bottom` and `handle_cab1_drawer_mid` are **handles**. They matched
  only because the word "drawer" appears in the name of the drawer they belong to.
- `cabinet2_door_out_fancy` and `cabinet3_door_bottom_out_fancy` are **doors** that slide
  out before they swing open. They are on prismatic joints, but you cannot open them like a
  drawer.

Neither 25 is the right answer. Hold on to that number — we will find out what the truth is
in §5, and it is neither of these.

And the name-based guess has a deeper problem than being wrong: it only works at all because
a human happened to name these links in English. Rename them `link_001 … link_113` — a
perfectly legal URDF, and what most CAD exporters give you — and it returns nothing.

The rest of this tutorial is about getting an answer that does not depend on either luck.

## 2. Saying what things are

A **semantic annotation** attaches meaning to bodies in a world: *this* body is a handle,
*these* bodies together are a drawer.

The library's position is worth stating plainly, because it is a design choice you may want
to argue with. Annotations are inspired by ontologies, but they are **not** an ontology:
there is no OWL, no RDF, no triple store, no separate reasoner. An annotation is an ordinary
Python dataclass, and reasoning is done with Python's type system plus a query language. The
claim is that you get the expressiveness without the impedance mismatch; the cost is that
your knowledge lives in Python rather than in a portable standard.

Here is a real one from the library:

```python
@dataclass(eq=False)
class Drawer(Furniture, HasCaseAsRootBody, HasHandle, HasMechanicalJoint):
    @classproperty
    def hole_direction(self) -> Vector3:
        return Vector3.Z()
```

The mixins *are* the definition: a drawer is furniture, it has a case as its root body, it
has a handle, and it has a mechanical joint. Remember that definition — in §5 it turns out
to be slightly too strict.

### Building a dresser

`create_with_new_body_in_world` spawns the annotation, a body, and its geometry in one call.
Then `add` wires the parts together — one method, routed by type.

In [ ]:
from semantic_digital_twin.world import World
from semantic_digital_twin.world_description.geometry import Scale
from semantic_digital_twin.spatial_types.spatial_types import (
    HomogeneousTransformationMatrix,
    Vector3,
)
from semantic_digital_twin.semantic_annotations.semantic_annotations import (
    Dresser,
    Drawer,
    Handle,
    Slider,
)

dresser_world = World.create_with_root_body()

with dresser_world.modify_world():
    dresser = Dresser.create_with_new_body_in_world(
        name="dresser",
        scale=Scale(0.6, 0.6, 0.5),
        world=dresser_world,
        world_root_T_self=HomogeneousTransformationMatrix(),
    )
    drawer = Drawer.create_with_new_body_in_world(
        name="drawer",
        scale=Scale(0.5, 0.5, 0.4),
        world=dresser_world,
        world_root_T_self=HomogeneousTransformationMatrix(),
    )
    handle = Handle.create_with_new_body_in_world(
        name="drawer_handle",
        world_root_T_self=HomogeneousTransformationMatrix.from_xyz_rpy(x=-0.28),
        world=dresser_world,
    )
    slider = Slider.create_with_new_body_in_world(
        name="drawer_slider",
        world_root_T_self=HomogeneousTransformationMatrix(),
        world=dresser_world,
        parent_connection_specification=Slider.parent_connection_specification(
            axis=Vector3.X()
        ),
    )

    # One method. It matches each part against the typed part-whole fields of the whole.
    drawer.add(handle)   # -> drawer.handle           (single-valued field)
    drawer.add(slider)   # -> drawer.mechanical_joint (single-valued field)
    dresser.add(drawer)  # -> dresser.drawers         (list field, appended)

print("drawer.handle           is handle:", drawer.handle is handle)
print("drawer.mechanical_joint is slider:", drawer.mechanical_joint is slider)
print("dresser.drawers                  :", len(dresser.drawers))

Note what `add` did *not* need: no `parent=`, no `child=`, no joint declaration. The part's
type was enough to decide both which field it belongs in and where it mounts in the
kinematic tree. You can see the tree it built:

In [ ]:
dresser_world.visualize_world_structure()

`visualize_world_structure` is the most useful debugging tool in the library — it draws the
kinematic tree as the robot actually sees it. (Try it on `world`, the apartment, if you
like: 113 bodies make a very wide image.)

### The payoff

The annotation is not a label sitting beside the geometry — it is wired into it. `Drawer`
has a `mechanical_joint`, so a drawer can be opened:

In [ ]:
ray_tracer = RayTracer(dresser_world)
ray_tracer.update_scene()
ray_tracer.scene.show("jupyter")

In [ ]:
drawer.mechanical_joint.position = 0.25   # metres along the slider axis

ray_tracer = RayTracer(dresser_world)
ray_tracer.update_scene()
ray_tracer.scene.show("jupyter")

The handle moved with the drawer front, because `add` made it a kinematic child. This is the
whole point: *"open the drawer"* went from an unanswerable question to one line of code, and
the geometry followed.

### Exercise 2

Add a second drawer to the dresser, above the first one. You need a `Drawer`, a `Handle` and
a `Slider`, wired with `add` exactly as above — the only things that change are the names
(they must be unique) and the `z` offset in `world_root_T_self`.

Then open it, by a different amount than the first, and render.

Assign your objects to `second_drawer`, `second_handle` and `second_slider`, so the check
below can find them.

In [ ]:
second_drawer: Drawer = ...
second_handle: Handle = ...
second_slider: Slider = ...

with dresser_world.modify_world():
    # TODO: create the three parts and wire them together with add()
    pass

# TODO: open it

In [ ]:
# Run this to check your answer.
from semantic_digital_twin.exceptions import ExerciseVerificationFailed

if second_drawer is ... or not isinstance(second_drawer, Drawer):
    raise ExerciseVerificationFailed("second_drawer should be a Drawer.")
if second_drawer.handle is not second_handle:
    raise ExerciseVerificationFailed("Use drawer.add(handle) to attach the handle.")
if second_drawer.mechanical_joint is not second_slider:
    raise ExerciseVerificationFailed("Use drawer.add(slider) to attach the slider.")
if len(dresser.drawers) != 2:
    raise ExerciseVerificationFailed("Use dresser.add(drawer) so the dresser has 2 drawers.")
if second_drawer.mechanical_joint.position == drawer.mechanical_joint.position:
    raise ExerciseVerificationFailed("Open the two drawers by different amounts.")

print("Correct.")
ray_tracer = RayTracer(dresser_world)
ray_tracer.update_scene()
ray_tracer.scene.show("jupyter")

### One detail that matters later

Annotations are declared `@dataclass(eq=False)`. That looks like boilerplate, but it is
load-bearing: the base class defines equality and hashing **structurally**, over the type and
the bodies referenced. Two separately constructed `Handle` objects on the same body are the
same handle:

In [ ]:
first = Handle(root=handle.root)
second = Handle(root=handle.root)

print("different objects:", id(first) != id(second))
print("but equal        :", first == second)
print("and same hash    :", hash(first) == hash(second))

Without this, the reasoner in §3 could not tell a newly inferred annotation from one it had
already found, and every run would pile up duplicates.

## 3. Not annotating by hand

Hand-annotating a dresser took twenty lines. The apartment has 113 bodies, and a building has
thousands. This does not scale, and it is not supposed to.

`WorldReasoner` applies a body of rules to a raw world and infers the annotations itself. It
runs on the apartment we loaded in §1 — the plain URDF, with nothing added.

In [ ]:
from semantic_digital_twin.reasoning.world_reasoner import WorldReasoner

reasoner = WorldReasoner(world)
inferred = reasoner.reason()["semantic_annotations"]

print(f"{len(inferred)} annotations inferred\n")
print(Counter(type(a).__name__ for a in inferred))

From a file that contained none of those words as *concepts*, the reasoner produced handles,
drawers, doors, and the cabinets they belong to.

So what does it say about Exercise 1?

In [ ]:
drawers = world.get_semantic_annotations_by_type(Drawer)

print(f"{len(drawers)} drawers\n")
for d in drawers:
    print("   ", d.root.name.name)

**Nineteen** — *fewer* than either heuristic's 25.

That is worth pausing on, because it looks like the reasoner did worse. It did not. The two
heuristics returned 25 bodies they could not justify; the reasoner returned 19 it can. It is
being conservative: it only claims what its rules actually support.

Which raises the obvious question, and the reason any of this is usable.

### Why does it think that is a drawer?

The reasoner is not a network. It can show its work.

In [ ]:
from krrood.entity_query_language.explanation.explanation import explain_inference
from krrood.entity_query_language.verbalization.pipeline import verbalize_expression

explanation = explain_inference(drawers[0])

print(explanation.get_satisfied_conditions_as_string())

In [ ]:
print(verbalize_expression(explanation.query_root))

Read that rule closely, because it is doing something neither heuristic in §1 could:

> If there's a FixedConnection whose parent is the child of a PrismaticConnection, there's a
> Handle whose root is the child of the FixedConnection, then there's a Drawer whose root is
> the parent of the FixedConnection, and whose handle is the Handle.

It is **structural**: "a body that slides, with a handle rigidly attached to it." Rename every
link to `link_042` and this rule still fires. That is also exactly why the two sliding doors
from §1 were excluded — they slide, but what is attached to them is not a handle.

The rules are not a black box and not a trained artifact. They are generated Python, checked
into the repository next to the code, so they are reviewed, versioned and migrated like
everything else:

In [ ]:
from semantic_digital_twin.reasoning import world_rdr

print(Path(world_rdr.__file__).parent)
for f in sorted(Path(world_rdr.__file__).parent.glob("*.py")):
    print("   ", f.name)

### An honest caveat

Not every rule is structural. The `Handle` rule, which the `Drawer` rule depends on, still
matches on the name — it looks for `"handle"` in the body name. In this apartment that
happens to be exactly right, all 29 of them, which is luck rather than design.

Keep both facts in mind: the drawer rule is structural, and it rests on a name-based one.

## 4. Asking the world questions

Now that the apartment carries meaning, we can query it. The **Entity Query Language** (EQL)
runs over plain Python objects — no database, no schema, no serialization step.

Three pieces: `variable` declares what you are quantifying over and from which collection,
`entity` says what you want back, and `an` / `the` evaluate it.

In [ ]:
from krrood.entity_query_language.factories import variable, entity, an, the, contains
from semantic_digital_twin.semantic_annotations.semantic_annotations import (
    Handle,
    Door,
    Wardrobe,
)

handle_variable = variable(Handle, world.semantic_annotations)
handles = list(an(entity(handle_variable)).evaluate())

print("handles:", len(handles))

`.where(...)` adds conditions. Conditions are written against the variable, and may walk its
attributes:

In [ ]:
drawer_variable = variable(Drawer, world.semantic_annotations)

query = an(entity(drawer_variable).where(
    contains(drawer_variable.root.name.name.lower(), "cabinet6")
))

for d in query.evaluate():
    print(d.root.name.name)

Use `the(...)` when you expect exactly one thing:

In [ ]:
some_drawer = the(entity(variable(Drawer, world.semantic_annotations))).first()
print(some_drawer.root.name.name)

### Queries that follow part-whole structure

The reasoner also inferred which cabinet each drawer belongs to, so we can ask questions that
span several objects:

In [ ]:
for w in world.get_semantic_annotations_by_type(Wardrobe):
    print(f"{w.root.name.name:12s} {len(w.drawers)} drawers")

### Opening a real drawer

Everything from §2 applies to the inferred annotations too — they are the same classes:

In [ ]:
target = the(entity(variable(Drawer, world.semantic_annotations))).first()
slider_connection = target.root.parent_connection

print("drawer ", target.root.name.name)
print("handle ", target.handle.root.name.name)
print("joint  ", type(slider_connection).__name__)
print("range  ", slider_connection.dof.limits.lower.position,
      "->", slider_connection.dof.limits.upper.position)

before = target.handle.root.global_pose.to_position().to_np().flatten()[:3]
slider_connection.position = slider_connection.dof.limits.upper.position
after = target.handle.root.global_pose.to_position().to_np().flatten()[:3]

print("moved  ", before, "->", after)

In [ ]:
ray_tracer = RayTracer(world)
ray_tracer.update_scene()
ray_tracer.scene.show("jupyter")

### Exercise 3

1. Write an EQL query returning every `Door` in the apartment; assign the result to `doors`.
2. Several cabinets tie for the most drawers. Find that maximum count, assign it to
   `most_drawers`, and assign the list of every wardrobe holding that many to `roomiest`.
3. Open every drawer in `roomiest` to its upper limit, and render the result.

The upper limit of a drawer's joint is
`drawer.root.parent_connection.dof.limits.upper.position`.

In [ ]:
doors: list = ...
most_drawers: int = ...
roomiest: list = ...

# TODO: 1. query the doors

# TODO: 2. find the maximum drawer count and every wardrobe that has it

# TODO: 3. open all of their drawers

In [ ]:
# Run this to check your answer.
if doors is ... or len(list(doors)) != 8:
    raise ExerciseVerificationFailed("There are 8 doors in this apartment.")
if not all(isinstance(d, Door) for d in doors):
    raise ExerciseVerificationFailed("doors should contain only Door annotations.")
if most_drawers != 3:
    raise ExerciseVerificationFailed("The largest number of drawers in one wardrobe is 3.")
if roomiest is ... or len(roomiest) != 5:
    raise ExerciseVerificationFailed("5 wardrobes tie for the most drawers.")
for w in roomiest:
    for d in w.drawers:
        connection = d.root.parent_connection
        if abs(connection.position - connection.dof.limits.upper.position) > 1e-6:
            raise ExerciseVerificationFailed("Every drawer in roomiest should be fully open.")

print("Correct.")
ray_tracer = RayTracer(world)
ray_tracer.update_scene()
ray_tracer.scene.show("jupyter")

The library also ships spatial predicates in
`semantic_digital_twin.reasoning.predicates` — `Above`, `Below`, `LeftOf`, `InsideOf`,
`is_supported_by`, `visible`, `reachable` — usable inside `.where(...)` the same way. Worth
exploring if you finish early.

## 5. Teaching the reasoner

Back in §1, 25 bodies slide on a prismatic joint. The reasoner reported 19 drawers. Six
sliding bodies are therefore not drawers, according to the rules. Let's see them.

In [ ]:
sliding_but_not_drawers = sliding - {d.root.name.name for d in drawers}
print(sorted(sliding_but_not_drawers))

Two of these we already know about — the `_out_fancy` doors from §1, correctly excluded.

The other four are named like drawers. Let's look at what is attached to each of them, and
compare with a drawer that *was* found:

In [ ]:
def children_of(body_name):
    body = world.get_body_by_name(body_name)
    children = body.child_kinematic_structure_entities
    if not children:
        return "   (nothing attached)"
    return "\n".join(
        f"   {c.name.name}  via {type(c.parent_connection).__name__}" for c in children
    )

for name in ["cabinet2_drawer_big", "coffee_table_drawer",
             "cabinet2_door_out_fancy", "cabinet6_drawer_top"]:
    print(f"{name}:")
    print(children_of(name))
    print()

There it is.

- `cabinet2_door_out_fancy` has a door attached on a **revolute** joint. It slides out, then
  swings open. Not a drawer — the reasoner is right.
- `cabinet6_drawer_top`, which *was* found, has a handle fixed to it.
- `cabinet2_drawer_big` and `coffee_table_drawer` have **nothing attached at all**. Whoever
  modelled this apartment did not give them a handle.

And that is the gap. Recall the definition from §2:

```python
class Drawer(Furniture, HasCaseAsRootBody, HasHandle, HasMechanicalJoint):
```

The rule implements exactly that: a drawer *has a handle*. These four slide, they sit inside
cabinets, a robot can open them by pulling — but no handle was modelled, so the rule cannot
see them. This is not a naming problem and no amount of better string matching would help.
The definition is a little too strict for the data.

So the truth for Exercise 1 is **23**: the 19 with handles, plus these 4 without. Neither
heuristic's 25 was right, and neither was the reasoner's 19.

### Ripple-Down Rules: fixing it without breaking the other 19

You do not edit the rule. You add a refinement, and the system asks you for it. This is the
Ripple-Down Rules workflow: the reasoner shows you its current answer, you say what is wrong,
and your correction is stored as a new rule attached to the case that triggered it. Existing
rules are never modified, so the 19 drawers that already work cannot regress.

> ### ⚠️ This part does not run in a notebook
>
> Fitting opens an interactive expert shell, and a notebook kernel has no terminal attached
> — you will get `NonInteractiveTerminalError`.
>
> **Open a terminal instead:** *File → New → Terminal* in JupyterLab.

In the terminal, start an IPython session on the CRAM interpreter:

```bash
cd ~/ijcai_2026_workshop
~/ijcai_2026_workshop/cognitive_robot_abstract_machine/.venv/bin/python -m IPython
```

Then paste this in:

```python
import logging; logging.disable(logging.CRITICAL)
from pathlib import Path
from importlib.resources import files
from semantic_digital_twin.adapters.urdf import URDFParser
from semantic_digital_twin.reasoning.world_reasoner import WorldReasoner
from semantic_digital_twin.semantic_annotations.semantic_annotations import Drawer

urdf = Path(files("semantic_digital_twin")).parent.parent / "resources" / "urdf" / "apartment.urdf"
world = URDFParser.from_file(str(urdf)).parse()

reasoner = WorldReasoner(world)
reasoner.fit_semantic_annotations([Drawer], update_existing_semantic_annotations=True)
```

You will be dropped into an expert shell showing the current value of
`World.semantic_annotations` for type `Drawer`. The commands are:

| command | what it does |
|---|---|
| `%help` | show the guide |
| `%edit` | open a template file for your rule |
| `%load` | load what you wrote and run it, so you can inspect the result |
| `%current_value` | show what the rule currently returns |
| accept | return the value to store it as a rule |
| `exit` | leave without changing anything |

`%edit` gives you a function stub to fill in:

```python
def world_semantic_annotations_of_type_drawer(case: World) -> List[Drawer]:
    """Get possible value(s) for World.semantic_annotations of type Drawer."""
    # Write code here
    pass
```

### Exercise 4 — write the rule

A rule is just a Python function returning a list. It does not have to be written in EQL, so
for this one plain Python is clearer.

Write a rule that finds the drawers that have no handle. Something counts if:

- it is the child of a `PrismaticConnection` — it slides; **and**
- none of its children is a `Handle` — otherwise the existing rule already has it; **and**
- none of its children is attached by a `RevoluteConnection` — that would make it one of the
  `_out_fancy` doors, which are not drawers.

Construct each result as `Drawer(root=body)`. You will want these imports, which the template
does not include:

```python
from semantic_digital_twin.world_description.connections import (
    PrismaticConnection,
    RevoluteConnection,
)
from semantic_digital_twin.semantic_annotations.semantic_annotations import Handle
```

`%load` it and check what it returns. Your rule is a *refinement*, so on its own it should
return exactly **4** drawers: `cabinet2_drawer_big`, `cabinet2_drawer_small`,
`cabinet8_drawer_middle` and `coffee_table_drawer`. If you get 6, you are catching the fancy
doors. If you get 23, you are re-deriving the drawers the existing rule already found.

Then accept the rule. It now sits alongside the original — the 19 drawers that already worked
cannot regress, because you added a rule rather than editing one. Re-run the reasoner on a
fresh world and you should get **23**:

```python
world = URDFParser.from_file(str(urdf)).parse()
WorldReasoner(world).reason()
len(world.get_semantic_annotations_by_type(Drawer))
```

Your rule was written to
`.../semantic_digital_twin/src/semantic_digital_twin/reasoning/world_rdr/` as ordinary
Python. Open the file and read it: that is the whole knowledge base, and it is now one rule
larger than when you started.

> If you get stuck, `exit` leaves everything untouched. The rules you write live inside your
> own container, so you cannot break anyone else's session.

## 6. Where to go next

What we did: took a URDF that knew only geometry, gave it a vocabulary of typed concepts, had
a rule base infer those concepts automatically, queried them, and then found a case the rules
got wrong and taught them the missing one. The robot can now be told "open the drawer".

The thing worth taking away is the last step. The heuristics in §1 were wrong and gave you
nothing to work with. The reasoner was also incomplete — but it could tell you exactly what
rule produced each answer, which is what made the gap findable and fixable in twenty lines.

What we skipped, and where to find it — the library ships a full Jupyter Book under
`cognitive_robot_abstract_machine/semantic_digital_twin/doc/` (17 worked examples, concept
chapters, and self-assessment quizzes):

| Topic | Guide |
|---|---|
| Transforms and the `A_T_B` convention | `examples/using_transformations.md` |
| Declarative world building | `examples/building_worlds_with_specifications.md` |
| Regions and supporting surfaces | `examples/regions.md` |
| Saving annotated worlds to SQL | `examples/persistence_of_annotated_worlds.md` |
| Physics simulation (MuJoCo) | `examples/physics_simulators.md` |
| Adding a new robot | `examples/adding_robots.md` |
| Free-space decomposition and path planning | `examples/graph_of_convex_sets.md` |
| Loading RoboCasa / ProcTHOR / PartNet scenes | `doc/datasets.md` |

To convert any of them into a runnable notebook:

```bash
jupytext --to notebook cognitive_robot_abstract_machine/semantic_digital_twin/examples/regions.md
```

The predecessor to this tutorial, on writing the URDF itself, is
[EASE Fall School 2024 — Creating an Environment URDF](https://github.com/IntEL4CoRo/ease_fall_school_2024).